# Why a robot is never sure: notebook

The hallway of the chapter: 10 cells of 1 m, numbered 0 to 9 and joined into a loop (one step right of cell 9 is cell 0), with doors at cells 1, 3 and 7. The command "move 2" moves the robot 1, 2 or 3 cells with probability 0.1, 0.8 and 0.1; the noisy door sensor reads "door" with probability 0.6 at a door and 0.2 at a wall. We check every number of the Note and simulate the robots ourselves.

In [1]:
import numpy as np
import pandas as pd

DOORS = [1, 3, 7]
N = 10
is_door = np.array([i in DOORS for i in range(N)])
MOVE = {1: 0.1, 2: 0.8, 3: 0.1}          # "move 2"
P_DOOR = np.where(is_door, 0.6, 0.2)     # P(reading "door" | cell)

## 1. Sensor noise and a passing person (Note, Section 2.2)

200 readings of a wall 2.00 m away: 190 with noise of standard deviation 3 cm, 10 while a person walks past.

In [2]:
rng = np.random.default_rng(7)
wall = rng.normal(2.0, 0.03, 190)
person = rng.uniform(0.6, 1.2, 10)
print('first readings:', np.round(wall[:3], 3))
print(f'wall readings from {wall.min():.2f} to {wall.max():.2f} m')
print(f'person readings from {person.min():.2f} to {person.max():.2f} m')

first readings: [2.    2.009 1.992]
wall readings from 1.92 to 2.07 m
person readings from 0.61 to 1.16 m


## 2. Dead reckoning without readings (Note, Section 2.1)

On a long straight track (no loop), each "move 2" goes 1 m (10 %), 2 m (80 %) or 3 m (10 %). We follow the exact probabilities, and check them with 100 000 simulated robots.

In [3]:
def predict_line(b):
    out = np.zeros(len(b))
    for i, p in enumerate(b):
        for d, q in MOVE.items():
            if i + d < len(b):
                out[i + d] += p * q
    return out

b = np.eye(25)[0]
rows = []
x = np.arange(25)
for n in range(1, 9):
    b = predict_line(b)
    sd = np.sqrt((b * (x - 2 * n) ** 2).sum())
    rows.append(dict(moves=n, robot_thinks_m=2 * n, P_exactly_there=round(b[2 * n], 3), spread_sd_m=round(sd, 3),
                     sd_from_variances_add=round(np.sqrt(0.2 * n), 3)))
pd.DataFrame(rows)

,moves,robot_thinks_m,P_exactly_there,spread_sd_m,sd_from_variances_add
0,1,2,0.800,0.447,0.447
1,2,4,0.660,0.632,0.632
2,3,6,0.560,0.775,0.775
3,4,8,0.487,0.894,0.894
4,5,10,0.432,1.000,1.000
5,6,12,0.391,1.095,1.095
6,7,14,0.358,1.183,1.183
7,8,16,0.332,1.265,1.265


In [4]:
sim = np.random.default_rng(0).choice([1, 2, 3], p=[0.1, 0.8, 0.1], size=(100_000, 8)).sum(axis=1)
print('simulated spread after 8 moves:', sim.std().round(3), ' share exactly at 16 m:', (sim == 16).mean().round(3))

simulated spread after 8 moves: 1.262  share exactly at 16 m: 0.333


## 3. One reading, three cells; a best guess gets lost (Note, Section 3)

Exact moves and a perfect sensor. Candidates after "door", after "move 2", and after "door" again.

In [5]:
cand = [i for i in range(N) if is_door[i]]
print('after z1 = door:', cand, ' each', round(1 / len(cand), 3))
moved = [(i + 2) % N for i in cand]
print('after u2 = move 2:', moved)
left = [i for i in moved if is_door[i]]
print('after z2 = door:', left)
print('chance a best guess picked the right door:', round(1 / len(cand), 3))

after z1 = door: [1, 3, 7]  each 0.333
after u2 = move 2: [3, 5, 9]
after z2 = door: [3]
chance a best guess picked the right door: 0.333


## 4. A noisy door sensor (Note, Section 3.3)

At a door the sensor reads "door" with probability 0.6, at a wall 0.2. Crossing out cells throws away the true cell whenever the reading is wrong; weighting keeps it.

In [6]:
w = P_DOOR / P_DOOR.sum()
print('shares: door', round(P_DOOR[1] / 0.2), ' wall', round(P_DOOR[0] / 0.2), ' total', round(P_DOOR.sum() / 0.2))
print('each door cell:', round(w[1], 4), ' each wall cell:', round(w[0], 4))
print('two readings at doors, at least one wrong:', round(1 - 0.6 ** 2, 2))
right = np.random.default_rng(1).random((100_000, 2)) < 0.6
print('simulated:', round(1 - right.all(axis=1).mean(), 3))

shares: door 3  wall 1  total 16
each door cell: 0.1875  each wall cell: 0.0625
two readings at doors, at least one wrong: 0.64
simulated: 0.638


## 5. Moving spreads, sensing sharpens (Note, Section 6)

Start surely in cell 3, two noisy moves of 2 cells, then the noisy sensor reads "door".

In [7]:
def predict(b):
    out = np.zeros(len(b))
    for i, p in enumerate(b):
        for d, q in MOVE.items():
            out[(i + d) % len(b)] += p * q
    return out

b = np.eye(N)[3]
b3 = predict(b)
b4 = predict(b3)
print('after u3:', np.round(b3, 3))
print('after u4:', np.round(b4, 3))
prod = b4 * P_DOOR
print('belief x P(door | cell), cells 5-9:', np.round(prod[5:10], 3), ' sum', round(prod.sum(), 3))
post = prod / prod.sum()
print('after z4 = door, cells 5-9:', np.round(post[5:10], 4))
x = np.arange(N)
for name, d in [('after u3', b3), ('after u4', b4), ('after z4', post)]:
    m = (d * x).sum()
    print(name, 'spread (sd):', round(np.sqrt((d * (x - m) ** 2).sum()), 3), 'm')

after u3: [0.  0.  0.  0.  0.1 0.8 0.1 0.  0.  0. ]
after u4: [0.   0.   0.   0.   0.   0.01 0.16 0.66 0.16 0.01]
belief x P(door | cell), cells 5-9: [0.002 0.032 0.396 0.032 0.002]  sum 0.464
after z4 = door, cells 5-9: [0.0043 0.069  0.8534 0.069  0.0043]
after u3 spread (sd): 0.447 m
after u4 spread (sd): 0.632 m
after z4 spread (sd): 0.415 m
